# 18 LNS / LNS2：大邻域搜索修解


本课按百科条目写。LNS 不从零证明最优，而是「已有可行解再抠」。调度台在最后。

---

## 1. 定义

**Large Neighborhood Search** 用于 MAPF（Li et al. 的 MAPF-LNS / LNS2）：

1. 用 HCA 等快速得到可行解（可很丑）。
2. **Destroy**：抽 k 台车（随机或按冲突/延迟）。
3. **Repair**：其余车路径当障碍，只重规划这 k 台。
4. 更好且可行则接受。重复。

LNS2 可以从 **仍有碰撞** 的路径集开始修到无碰撞。本仓库从可行 HCA 出发，保证历史 SOC 非增。

- 输入：图、起终点、轮数。
- 输出：路径、SOC、`history` 列表。

---

## 2. 保证

无最优、无完备。经验上 SOC 非增（只接受更好或持平）。适合「已经能跑再抠质量」。k 太大 = 重跑 HCA；k=1 邻域太小，改善慢。

---

## 3. 不变量

- 接受条件：新路径无冲突 **且** SOC ≤ 当前。
- `history` 单调非增，给 eval 用。
- frozen 车写入占用表，subset 车看不见彼此以外的自由——它们按某种序时间 A\*。

---

## 4. 伪代码

```
current ← HCA(...)
for round:
    subset ← 随机 k 车
    frozen ← 其余车路径写入 table
    按某种序对 subset 做 TimeA*
    若无冲突且 SOC ≤ current: current ← 新路径
```

---

## 5. 手算

两车平行、HCA 已经最优。Destroy 任一台再修，SOC 不能再降。`history` 应全是同一个数。改善出现在「HCA 顺序很差、绕路很多」的实例。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：一轮 destroy / repair


In [ ]:
from evals.scenarios import two_agent_tunnel
from lib.algos.prioritized import hca, _reserve_path
from lib.reservation import ReservationTable
from lib.algos.temporal import time_astar
from lib.agents import is_conflict_free, soc


def one_repair(graph, starts, goals, current_paths, subset):
    frozen = {a: p for a, p in current_paths.items() if a not in subset}
    table = ReservationTable()
    for a, p in frozen.items():
        _reserve_path(table, p, a)
    new_paths = dict(frozen)
    for a in subset:
        r = time_astar(graph, starts[a], goals[a], table=table, agent=a)
        print("  重规划", a, "found", r.found, "cost", r.cost)
        if not r.found:
            return None
        new_paths[a] = r.path
        _reserve_path(table, r.path, a)
    if is_conflict_free(new_paths):
        return new_paths
    return None


g, starts, goals = two_agent_tunnel()
cur = hca(g, starts, goals)
print("初始 HCA SOC", cur["soc"])
repaired = one_repair(g, starts, goals, cur["paths"], ["a"])
if repaired:
    print("修 a 后 SOC", soc(repaired), "非增", soc(repaired) <= cur["soc"])
else:
    print("本轮修失败，保留原解")


In [ ]:
from lib.algos.cbs import lns

r = lns(g, starts, goals, rounds=8)
print("found", r["found"], "SOC", r["soc"], "历史", r["history"])
print("历史非增", all(r["history"][i] <= r["history"][i - 1] for i in range(1, len(r["history"]))))


## 7. 逐行

| 块 | 作用 |
|----|------|
| 初始 HCA 失败则反序再试 | 库提高可行率 |
| `frozen` occupy | 邻域外当移动墙 |
| 只接受 ≤ | anytime 单调 |
| `history` | eval 检查非增 |

## 常见 bug

1. 接受更差解却不记录（破坏 anytime 单调）。
2. destroy 抽全部车 = 重跑 HCA，邻域太大。
3. frozen 不锁边，repair 与 frozen 对向擦肩。
4. 从不可行解出发却用「SOC≤」当唯一接受条件（LNS2 要先把冲突数降下来）。

## 条目小结

| 项目 | 内容 |
|------|------|
| 思想 | destroy + repair |
| 保证 | 无；SOC 非增 |
| 下一课 | 不搜树，按规则推开挡路者 |

## 练习

1. k=1 与 k=n 的邻域各是什么？
2. 为何随机种子要固定才能让 eval 可重复？
3. 把接受条件改成「无冲突即可、允许 SOC 变差」，history 还单调吗？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_multi(g.to_studio(), r, "LNS")
studio
